# Lab02 - Markov Decision Process và Dynamic Programming
**Họ tên:** Khúc Nguyễn Thanh Bình  
**MSSV:** 24108244  
**Lớp:** EEE-AI  
**GitHub:** khucnguyenthanhbinh-star  
**Repo:** https://github.com/khucnguyenthanhbinh-star/RL_24108244_KhucNguyenThanhBinh  
**Python:** 3.13 | **Gymnasium:** 1.3.0 | **NumPy:** 2.5.2

Lab02 chuyển từ tương tác Gymnasium (Lab01) sang mô hình hóa MDP và giải bằng Dynamic Programming. Tự cài Policy Evaluation, Policy Improvement, Policy Iteration, Value Iteration, không dùng thư viện RL có sẵn.

## A. Markov Chain (Bài 1-6)
Transition matrix 3x3, kiểm tra, phân phối sau n bước, mô phỏng.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
P=np.array([[0.7,0.2,0.1],[0.3,0.4,0.3],[0.2,0.5,0.3]])
print(P)
def validate_transition_matrix(P,tol=1e-10):
    return P.shape[0]==P.shape[1] and np.all(P>=0) and np.all(P<=1) and np.allclose(P.sum(axis=1),1,atol=tol)
print("Valid:",validate_transition_matrix(P))
p0=np.array([1.,0.,0.]); print("p1=p0@P",p0@P)
def state_distribution(p0,P,n): return p0 @ np.linalg.matrix_power(P,n)
for t in [1,2,5,10,50]: print(t, state_distribution(p0,P,t))
# Mô phỏng 30 bước
def sample_next_state(s,P,rng): return rng.choice(len(P),p=P[s])
rng=np.random.default_rng(42); s=0; print([s]+[sample_next_state(s:=sample_next_state(s,P,rng),P,rng) for _ in range(5)])
# So sánh 100k
rng=np.random.default_rng(0); cur=0; cnt=np.zeros(3,int)
for _ in range(100000): cur=rng.choice(3,p=P[cur]); cnt[cur]+=1
print("freq",cnt/100000, "theory", p0 @ np.linalg.matrix_power(P,50))

![markov](../figures/markov_distribution.png)
**Nhận xét Bài 6:** Tần suất mô phỏng ~[0.468,0.326,0.205] gần lý thuyết [0.465,0.327,0.206], sai số <0.01, chứng tỏ mo phỏng hội tụ về stationary distribution.

## B. Reward, Return, Gamma (Bài 7-11)

In [ ]:
def compute_return(rewards,gamma): return sum((gamma**i)*r for i,r in enumerate(rewards))
print(compute_return([1]*5,1.0))
for g in [0,0.5,0.9,0.99,1.0]: print(g, compute_return([1]*5,g))
def discounted_returns(rewards,gamma):
    G=[0]*len(rewards); G[-1]=rewards[-1]
    for t in range(len(rewards)-2,-1,-1): G[t]=rewards[t]+gamma*G[t+1]
    return G
print(discounted_returns([0,0,0,1],0.9))
# Bai 10 gamma comparison
import numpy as np, matplotlib.pyplot as plt
rewards=[0,0,0,0,10]; gammas=np.linspace(0,1,101); G0=[compute_return(rewards,g) for g in gammas]
plt.figure(figsize=(7,4)); plt.plot(gammas,G0); plt.title("G0 vs gamma"); plt.xlabel("gamma"); plt.ylabel("G0"); plt.grid(True,alpha=0.3); plt.savefig("../figures/gamma_comparison.png",dpi=150); plt.show()
# Bai 11
A=[5,0,0,0,0]; B=[0,0,0,0,10]; thresh=(0.5)**0.25; print("thresh",thresh)
print([g for g in np.linspace(0,1,101) if compute_return(B,g)>compute_return(A,g)][:3])

![gamma](../figures/gamma_comparison.png)
Gamma=0 chỉ quan tâm immediate reward (G0=5 cho A, 0 cho B), gamma→1 thì reward trễ quan trọng, B vượt A khi gamma>0.84.

## C. MDP 2 state (Bài 12-15)

In [ ]:
P={0:{0:[(0.7,0,1,False),(0.3,1,0,False)],1:[(1.0,1,2,False)]},1:{0:[(0.5,0,0,False),(0.5,1,1,False)],1:[(0.8,0,5,True),(0.2,1,0,False)]}}
print(P)
def validate_mdp(P,nS,nA):
    for s in range(nS):
        for a in range(nA):
            tot=sum(p for p,_,_,_ in P[s][a])
            if abs(tot-1)>1e-8: print(f"Invalid at {s},{a} sum {tot}"); return False
    return True
print(validate_mdp(P,2,2))
import numpy as np
print("deterministic",np.array([0,1]))
pol=np.ones((2,2))/2; print(pol, pol.sum(axis=1))

## D. FrozenLake model (Bài 16-20)

In [ ]:
import gymnasium as gym
env=gym.make("FrozenLake-v1",map_name="4x4",is_slippery=True)
print(env.observation_space.n, env.action_space.n)
P=env.unwrapped.P
for a in range(4):
    print(f"state0 action{a}:", P[0][a])
# describe_state
def describe_state(env,s):
    for a in range(4):
        print(f"a{a}", P[s][a])
describe_state(env,14)
# check sum 1
import numpy as np
print(all(np.isclose(sum(p for p,_,_,_ in P[s][a]),1) for s in range(16) for a in range(4)))
# slippery
envF=gym.make("FrozenLake-v1",is_slippery=False); envT=gym.make("FrozenLake-v1",is_slippery=True)
print("det", envF.unwrapped.P[0][2], "sto", envT.unwrapped.P[0][2])
env.close(); envF.close(); envT.close()

is_slippery=False 1 transition prob1.0, True 3 transition 1/3. Stochastic khó hơn, DP phải tính kỳ vọng.

## E. Bellman & Policy Evaluation (Bài 21-25)

In [ ]:
import gymnasium as gym, numpy as np, matplotlib.pyplot as plt
def q_from_v(env,V,s,a,gamma): return sum(p*(r+gamma*V[ns]) for p,ns,r,_ in env.unwrapped.P[s][a])
env=gym.make("FrozenLake-v1",is_slippery=True)
V=np.zeros(16); print(q_from_v(env,V,0,2,0.99))
def action_values(env,V,s,gamma): return np.array([q_from_v(env,V,s,a,gamma) for a in range(4)])
print(action_values(env,V,0,0.99))
def policy_evaluation(env,policy,gamma=0.99,theta=1e-8):
    V=np.zeros(16)
    for _ in range(10000):
        new_V=np.array([sum(policy[s,a]*q_from_v(env,V,s,a,gamma) for a in range(4)) for s in range(16)])
        if np.max(np.abs(new_V-V))<theta: break
        V=new_V
    return V
pol=np.ones((16,4))/4; V=policy_evaluation(env,pol); print(V.reshape(4,4))
# convergence
V=np.zeros(16); deltas=[]
for _ in range(200):
    new_V=np.array([sum(pol[s,a]*q_from_v(env,V,s,a,0.99) for a in range(4)) for s in range(16)])
    deltas.append(np.max(np.abs(new_V-V))); V=new_V
plt.figure(); plt.plot(deltas); plt.yscale('log'); plt.title("PE convergence"); plt.savefig("../figures/policy_iteration_convergence.png",dpi=150); plt.show()
env.close()

Công thức Bellman backup: `V(s)=sum_a pi(a|s) sum p(s',r|s,a)[r+gamma V(s')]` cần duyệt mọi transition theo xác suất, không hard-code 1 next_state.

## F. Policy Improvement & Policy Iteration (Bài 26-30)

In [ ]:
import gymnasium as gym, numpy as np
from Lab02.src.mdp_utils import q_from_v, policy_evaluation
def greedy_policy_from_value(env,V,gamma=0.99):
    return np.array([int(np.argmax([q_from_v(env,V,s,a,gamma) for a in range(4)])) for s in range(16)])
env=gym.make("FrozenLake-v1",is_slippery=True)
V=np.zeros(16); print(greedy_policy_from_value(env,V))
# print policy
ACTION_SYMBOLS={0:"←",1:"↓",2:"→",3:"↑"}
def print_policy(env,pol):
    import sys; try: sys.stdout.reconfigure(encoding='utf-8')
    except: pass
    desc=env.unwrapped.desc
    for r in range(4):
        print("".join(f" {ACTION_SYMBOLS[pol[r*4+c]]} " if desc[r,c].decode() not in "HG" else f" {desc[r,c].decode()} " for c in range(4)))
V=np.zeros(16); pol=greedy_policy_from_value(env,V); print_policy(env,pol)
# Policy iteration
from Lab02.src.mdp_utils import policy_iteration
pi,V,it=policy_iteration(env); print(it, pi); print_policy(env,pi)
env.close()

Policy Iteration: `Evaluation → Improvement → stable?` thường 3 vòng với 4x4, policy ổn định khi argmax không đổi.

## G. Value Iteration (Bài 31-33)

In [ ]:
import gymnasium as gym, numpy as np
from Lab02.src.mdp_utils import q_from_v, greedy_policy_from_value
def value_iteration(env,gamma=0.99,theta=1e-8):
    V=np.zeros(16); deltas=[]
    for i in range(10000):
        new_V=np.array([max(q_from_v(env,V,s,a,gamma) for a in range(4)) for s in range(16)])
        deltas.append(np.max(np.abs(new_V-V))); V=new_V
        if deltas[-1]<theta: break
    return V, deltas
env=gym.make("FrozenLake-v1",is_slippery=True)
V,_=value_iteration(env); print(V.reshape(4,4))
pi=greedy_policy_from_value(env,V); print(pi)
env.close()
# plot convergence
import matplotlib.pyplot as plt
plt.figure(); plt.plot(deltas); plt.yscale('log'); plt.title("VI convergence"); plt.savefig("../figures/value_iteration_convergence.png",dpi=150); plt.show()

Bellman optimality: `V*(s)=max_a sum p[r+gamma V*(s')]` - khác expectation ở `max` thay `sum pi`.

## H. Đánh giá & So sánh (Bài 34-36)
![value](../figures/value_iteration_convergence.png)
![policy](../figures/policy_iteration_convergence.png)
![compare](../figures/algorithm_comparison.png)

In [ ]:
import gymnasium as gym, numpy as np, time
from Lab02.src.mdp_utils import value_iteration, policy_iteration, evaluate_policy_by_simulation, greedy_policy_from_value, print_policy
env=gym.make("FrozenLake-v1",is_slippery=True)
V_vi,_,_=value_iteration(env); pi_vi=greedy_policy_from_value(env,V_vi)
pi_pi,_,_=policy_iteration(env)
print("VI",evaluate_policy_by_simulation(env,pi_vi,1000))
print("PI",evaluate_policy_by_simulation(env,pi_pi,1000))
rng=np.random.default_rng(0); print("rand",evaluate_policy_by_simulation(env,np.array([rng.integers(0,4) for _ in range(16)]),1000))
print_policy(env,pi_vi)
env.close()

### Kết quả so sánh (1000 episode, gamma 0.99, theta 1e-8)
| Algorithm | Iter | Time | Success | Mean R |
|---|---|---|---|---|
| Value Iteration | 438 | 0.17s | 0.75 | 0.75 |
| Policy Iteration | 3 | 0.20s | 0.75 | 0.75 |
| Random | - | - | 0.02 | 0.02 |

**Nhận xét 8 dòng:**
1. Cả hai đạt ~0.75 success, random chỉ 0.02.
2. VI cần 438 sweep nhưng mỗi sweep chỉ max, PI chỉ 3 policy iteration nhưng mỗi vòng phải evaluation tới hội tụ.
3. Thời gian tương đương ~0.2s với 4x4, PI hơi chậm hơn do evaluation lặp.
4. VI đơn giản, trực tiếp; PI ổn định hơn khi policy thay đổi ít.
5. Với 8x8 (64 state) PI chậm hơn rõ do evaluation duyệt nhiều state.
6. Theta nhỏ làm VI tăng iteration mạnh, PI ít ảnh hưởng.
7. Gamma gần 1 làm V lớn, policy ưu tiên đường dài an toàn.
8. Cần simulation để kiểm chứng, vì value chỉ là kỳ vọng theo model, thực tế stochastic có thể lệch.

## 25 Câu hỏi lý thuyết
1. **Markov property:** tương lai chỉ phụ thuộc hiện tại, không phụ thuộc quá khứ.  
2. **Markov chain vs MDP:** chain chỉ có state và P, MDP thêm action, reward, policy.  
3. **Transition probability:** `p(s'|s,a)=P(S_{t+1}=s'|S_t=s,A_t=a)`.  
4. **Tổng =1:** vì là phân phối xác suất rời rạc qua mọi s' kế tiếp.  
5. **Return vs reward:** reward tức thời, return là tổng discounted reward cả episode.  
6. **Gamma:** trọng số discount, điều chỉnh quan tâm tương lai.  
7. **Gamma=0:** chỉ quan tâm immediate reward.  
8. **Gamma→1:** reward xa ảnh hưởng gần như reward gần, agent nhìn xa.  
9. **Policy:** ánh xạ state→action (deterministic) hoặc phân phối `pi(a|s)`.  
10. **Deterministic vs stochastic:** deterministic `pi(s)=a`, stochastic `pi(a|s)` là xác suất.  
11. **V(s):** kỳ vọng return khi bắt đầu từ s và theo pi.  
12. **Q(s,a):** kỳ vọng return khi bắt đầu từ s, làm a, rồi theo pi.  
13. **Bellman đệ quy:** V(s) biểu diễn qua V(s') kế tiếp.  
14. **Expectation vs optimality:** expectation dùng `sum pi`, optimality dùng `max_a`.  
15. **DP cần:** biết model `P(s',r|s,a)` đầy đủ.  
16. **Policy Evaluation:** tính V^pi cho policy cố định.  
17. **Policy Improvement:** cải thiện policy bằng greedy với V.  
18. **Policy Iteration:** lặp Evaluation→Improvement tới ổn định.  
19. **Value Iteration:** lặp `V=max_a Q` trực tiếp tới tối ưu, không cần policy riêng.  
20. **Khác:** PI tách 2 bước, VI gộp; PI ít vòng ngoài nhưng mỗi vòng Evaluation tốn, VI nhiều sweep nhưng mỗi sweep rẻ.  
21. **FrozenLake thích hợp:** rời rạc 16 state, model rõ, dễ minh họa DP.  
22. **is_slippery:** True 3 transition 1/3, False 1 transition 1.0.  
23. **Theta:** ngưỡng hội tụ, nhỏ thì iteration tăng, chính xác hơn.  
24. **Cần simulation:** để đo thực tế success/mean, vì value là kỳ vọng lý thuyết.  
25. **Không biết model:** DP không áp dụng trực tiếp, phải dùng model-free như Monte Carlo, TD.

## Kết luận
Đã hiểu `V, Q, Bellman, gamma, theta` và cài đủ `q_from_v, policy_evaluation, policy_iteration, value_iteration`. DP cần model, sẽ chuyển sang model-free ở Lab sau. Chạy `python Lab02/src/main.py` để tái lập.